# 🧠 Train COOLANT on VifactCheck with Anti-Overfitting

**Train COOLANT model on Vietnamese fake news HDF5 data with best practices**

📂 Input: `../processed_data/hdf5/vifactcheck_*.h5`

**Improvements over baseline:**

-   Fixed fake label generation: `make_pair_sim` now produces real (1) / fake (0) labels via image-swap — detection task is no longer trivially 100% accurate
-   `HDF5Dataset` returns `(text, image)` only — labels generated at batch time
-   Full-batch pair generation: all B samples used as anchors (not batch//2), doubling detection signal to 2B per step
-   Hard (30%) + random (70%) image-swap negatives for diverse fake pairs
-   Single `make_pair_sim` call feeds both similarity task (triplet) and detection task (2B labeled batch)
-   Early stopping with patience=5
-   Cosine annealing LR scheduler with warmup
-   Increased dropout (0.3) in FastCNN
-   Gradient clipping (max_norm=1.0)
-   Label smoothing (0.1) for detection
-   Weight decay (1e-4)


In [ ]:
# Setup and imports
import sys
import os
import math
import random
import json
import warnings
import datetime
from pathlib import Path
import mlflow
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import h5py
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

warnings.filterwarnings("ignore")

# Project path
project_root = (
    Path().absolute().parent.parent
)  # Go up 2 levels from notebooks/research/
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))  # Also add src directly

# Verify src is accessible
try:
    import src

    print(f"✅ src module found at: {src.__file__}")
except ImportError as e:
    print(f"❌ src module not found: {e}")

print(f"✅ Project root: {project_root}")


def detect_runtime() -> str:
    """Return 'colab' if running inside Google Colab, otherwise 'local'."""
    try:
        import google.colab  # type: ignore
        from google.colab import drive

        drive.mount("/content/drive")
        return "colab"
    except ImportError:
        return "local"


def resolve_paths(root: Path):
    """Compute base, data, and checkpoint paths for the current runtime."""
    runtime = detect_runtime()
    if runtime == "colab":

        base = Path(
            os.environ.get(
                "COLAB_BASE_DIR",
                "/content/drive/MyDrive/Thesis_Final/fake-new-detection",
            )
        )
    else:
        local_override = os.environ.get("LOCAL_BASE_DIR")
        base = Path(local_override) if local_override else root

    hdf5_dir = base / "notebooks" / "processed_data" / "hdf5"
    save_dir = base / "training" / "checkpoints"

    # Ensure directories exist for whichever runtime we're using
    for path in (hdf5_dir, save_dir):
        path.mkdir(parents=True, exist_ok=True)

    return runtime, base, hdf5_dir, save_dir


def ensure_colab_prereqs(runtime: str):
    """Install missing pip packages when running inside Colab."""
    if runtime != "colab":
        return

    import subprocess
    import sys

    packages = ["mlflow"]
    print(f"⬇️  Installing Colab prerequisites: {packages}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--upgrade"] + packages, check=True
    )
    print("✅ Colab prerequisites installed")

: 

In [ ]:
import mlflow

In [ ]:
# Device, seeds, and runtime paths
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"🔧 Device: {DEVICE}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Paths (auto-switch between local/Colab without manual edits)
runtime, BASE_DIR, HDF5_DIR, SAVE_DIR = resolve_paths(project_root)
ensure_colab_prereqs(runtime)

# Import mlflow after ensuring prerequisites are installed
import mlflow
import mlflow.pytorch

print(f"🌐 Runtime: {runtime} → base_dir={BASE_DIR}")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 HDF5 dir: {HDF5_DIR}")
print(f"💾 Save dir: {SAVE_DIR}")

In [ ]:
if runtime == "colab":
    from google.colab import drive

    drive.mount("/content/drive", force_remount=True)

In [ ]:
# Device, seeds, and runtime paths
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"🔧 Device: {DEVICE}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Paths (auto-switch between local/Colab without manual edits)
runtime, BASE_DIR, HDF5_DIR, SAVE_DIR = resolve_paths(project_root)
print(f"🌐 Runtime: {runtime} → base_dir={BASE_DIR}")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 HDF5 dir: {HDF5_DIR}")
print(f"💾 Save dir: {SAVE_DIR}")

In [ ]:
# Import HDF5Dataset from src
try:
    from processing.hdf5_dataset import HDF5Dataset as HDF5DatasetFull

    print("✅ Imported HDF5Dataset from processing.hdf5_dataset")
except ImportError:
    try:
        from src.processing.hdf5_dataset import HDF5Dataset as HDF5DatasetFull

        print("✅ Imported HDF5Dataset from src.processing.hdf5_dataset")
    except ImportError as e:
        print(f"❌ Could not import HDF5Dataset: {e}")
        print("Defining simplified version inline (file kept open)...")

        # Fallback: keep file open for the lifetime of the dataset object
        # Avoids repeated open/close on every __getitem__ call (critical for Drive I/O)
        class HDF5DatasetFull(Dataset):
            def __init__(self, hdf5_path):
                self.hdf5_path = hdf5_path
                self._file = h5py.File(self.hdf5_path, "r")
                self.n_samples = len(self._file["text_features"])
                print(f"✅ HDF5Dataset: {self.n_samples} samples from {hdf5_path.name}")

            def __len__(self):
                return self.n_samples

            def __getitem__(self, idx):
                text = self._file["text_features"][idx]
                image = self._file["image_features"][idx]
                text_tensor = torch.from_numpy(text).float().transpose(0, 1)
                image_tensor = torch.from_numpy(image).float()
                return text_tensor, image_tensor

            def close(self):
                if hasattr(self, "_file") and self._file:
                    self._file.close()
                    self._file = None

            def __del__(self):
                self.close()


# Wrapper that returns only (text, image), ignoring labels from the full dataset
class HDF5Dataset:
    """Wrapper that returns (text, image) only, no labels."""

    def __init__(self, hdf5_path):
        self.dataset = HDF5DatasetFull(hdf5_path)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        result = self.dataset[idx]
        if isinstance(result, tuple) and len(result) == 3:
            return result[0], result[1]
        return result

    def close(self):
        if hasattr(self.dataset, "close"):
            self.dataset.close()


print("✅ HDF5Dataset ready (file kept open — no repeated open/close per sample)")

In [ ]:
# Verification: confirm runtime-aware paths
print(f"🧭 Detected runtime: {runtime}")
print(f"   HDF5 directory: {HDF5_DIR}")
print(f"   Checkpoints directory: {SAVE_DIR}")

In [ ]:
# Import HDF5DatasetSimple from src
try:
    from processing.hdf5_dataset import HDF5DatasetSimple

    print("✅ Imported HDF5DatasetSimple from processing.hdf5_dataset")
except ImportError:
    try:
        from src.processing.hdf5_dataset import HDF5DatasetSimple

        print("✅ Imported HDF5DatasetSimple from src.processing.hdf5_dataset")
    except ImportError as e:
        print(f"❌ Could not import HDF5DatasetSimple: {e}")
        print("Defining simplified version inline (file kept open)...")

        # Fallback: keep file open for the lifetime of the dataset object
        # Avoids repeated open/close on every __getitem__ call (critical for Drive I/O)
        class HDF5DatasetSimple(Dataset):
            def __init__(self, hdf5_path):
                self.hdf5_path = hdf5_path
                self._file = h5py.File(self.hdf5_path, "r")
                self.n_samples = len(self._file["text_features"])
                print(
                    f"✅ HDF5DatasetSimple: {self.n_samples} samples from {hdf5_path.name}"
                )

            def __len__(self):
                return self.n_samples

            def __getitem__(self, idx):
                text = self._file["text_features"][idx]
                image = self._file["image_features"][idx]
                text_tensor = torch.from_numpy(text).float().transpose(0, 1)
                image_tensor = torch.from_numpy(image).float()
                return text_tensor, image_tensor

            def close(self):
                if hasattr(self, "_file") and self._file:
                    self._file.close()
                    self._file = None

            def __del__(self):
                self.close()


# Alias for consistency with notebook code
HDF5Dataset = HDF5DatasetSimple

print("✅ HDF5Dataset ready (file kept open — no repeated open/close per sample)")

In [ ]:
# Create DataLoaders
BATCH_SIZE = 32

datasets = {}
loaders = {}
missing_files = []

for split in ["train", "dev", "test"]:
    h5_path = HDF5_DIR / f"vifactcheck_{split}.h5"
    print(f"\n🔧 Loading {split}...")

    if h5_path.exists():
        dataset = HDF5Dataset(h5_path)
        datasets[split] = dataset

        shuffle = split == "train"
        loader = DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=shuffle,
            num_workers=0,  # Required for HDF5
            pin_memory=True if DEVICE.type == "cuda" else False,
        )
        loaders[split] = loader
        print(f"✅ {split}: {len(loader)} batches")
    else:
        print(f"❌ File not found: {h5_path}")
        missing_files.append(h5_path)

if missing_files:
    print(f"\n{'='*60}")
    print("⚠️  MISSING HDF5 FILES")
    print(f"{'='*60}")
    print("\nThe following HDF5 files are required but not found:")
    for f in missing_files:
        print(f"  - {f.name}")
    print(f"\n📂 Expected location: {HDF5_DIR}")
    print(
        "\n💡 SOLUTION: Run notebooks/research/6_multimodal_hdf5_preprocessing.ipynb first."
    )
    raise FileNotFoundError(
        f"Missing {len(missing_files)} HDF5 file(s). Run preprocessing first."
    )

# Verify shapes (dataset returns 2-tuple: text, image)
if "train" in loaders:
    sample_text, sample_image = datasets["train"][0]
    print(f"\n📏 Sample shapes:")
    print(f"  Text:  {sample_text.shape} (expected: [768, 512])")
    print(f"  Image: {sample_image.shape} (expected: [2048])")
    print(
        f"\n✅ All DataLoaders ready! Labels generated at batch time via make_pair_sim."
    )
else:
    raise RuntimeError("No training data available")

In [ ]:
# Import COOLANT - use direct import with runtime-aware paths
import sys
from pathlib import Path

# Use the BASE_DIR that was already resolved correctly in earlier cells
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
if str(BASE_DIR / "src") not in sys.path:
    sys.path.insert(0, str(BASE_DIR / "src"))

print(f"🔍 Python path includes:")
print(f"   {BASE_DIR}")
print(f"   {BASE_DIR / 'src'}")

try:
    import models.coolant_official as coolant
    from models.base import FastCNN

    print("✅ Imported models directly")
except ImportError:
    try:
        import src.models.coolant_official as coolant
        from src.models.base import FastCNN

        print("✅ Imported from src.models")
    except ImportError as e:
        print(f"❌ Import failed: {e}")
        print(f"❌ sys.path: {sys.path[:3]}")
        raise

IMAGE_DIM = 2048
TEXT_EMBED = 768
TEXT_SEQ_LEN = 512
CONFIG = {
    "shared_dim": 128,
    "sim_dim": 64,
    "clip_embed_dim": 64,
    "feature_dim": 64
    + 16
    + 16,  # 96 — default; patch_unimodal in cell 6 keeps this consistent
    "h_dim": 64,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "max_epochs": 30,
    "patience": 10,
    "dropout": 0.3,
    "label_smoothing": 0.1,
    "grad_clip": 1.0,
    "seed": SEED,
    "device": str(DEVICE),
    "batch_size": BATCH_SIZE,
}


class ResNetCOOLANT(coolant.COOLANT_Official):
    """COOLANT for ResNet50 (2048-dim) and PhoBERT (768-dim)."""

    def encode_text(self, text):
        dummy_img = torch.zeros(text.size(0), IMAGE_DIM, device=text.device)
        t, _ = self.similarity_module.encoding(text, dummy_img)
        return t

    def encode_image(self, image):
        dummy_txt = torch.zeros(
            image.size(0), TEXT_EMBED, TEXT_SEQ_LEN, device=image.device
        )
        _, i = self.similarity_module.encoding(dummy_txt, image)
        return i

    def fuse_modalities(self, text_f, image_f):
        return torch.cat([text_f, image_f], dim=-1)


print("✅ ResNetCOOLANT class defined")
print(
    f"   feature_dim={CONFIG['feature_dim']}, h_dim={CONFIG['h_dim']}, patience={CONFIG['patience']}"
)

In [ ]:
# Create and patch model
model = ResNetCOOLANT(CONFIG)


def patch_encoding(enc):
    """Patch shared_image from 512 -> IMAGE_DIM."""
    layers, done = [], False
    for l in enc.shared_image:
        if isinstance(l, nn.Linear) and not done:
            layers.append(nn.Linear(IMAGE_DIM, l.out_features))
            done = True
        else:
            layers.append(l)
    enc.shared_image = nn.Sequential(*layers)


# Patch encodings (only similarity and detection — NOT ambiguity_module.encoding
# because AmbiguityLearning.forward() never calls self.encoding)
patch_encoding(model.similarity_module.encoding)
patch_encoding(model.detection_module.encoding)


def patch_clip_projection(clip_module, is_image=True):
    """Patch CLIP projection to correct input dim."""
    target_dim = IMAGE_DIM if is_image else TEXT_EMBED
    proj = clip_module.image_projection if is_image else clip_module.text_projection

    layers, done = [], False
    for l in proj:
        if isinstance(l, nn.Linear) and not done:
            layers.append(nn.Linear(target_dim, l.out_features))
            done = True
        else:
            layers.append(l)

    new_proj = nn.Sequential(*layers)
    if is_image:
        clip_module.image_projection = new_proj
    else:
        clip_module.text_projection = new_proj


# Patch CLIP projections
patch_clip_projection(model.clip_module, is_image=True)
patch_clip_projection(model.clip_module, is_image=False)


def patch_unimodal(uni_module, prime_dim):
    """Patch UnimodalDetection output dim to match feature_dim."""
    shared_dim = 128  # fixed in EncodingPart
    uni_module.text_uni = nn.Sequential(
        nn.Linear(shared_dim, shared_dim),
        nn.BatchNorm1d(shared_dim),
        nn.ReLU(),
        nn.Linear(shared_dim, prime_dim),
        nn.BatchNorm1d(prime_dim),
        nn.ReLU(),
    )
    uni_module.image_uni = nn.Sequential(
        nn.Linear(shared_dim, shared_dim),
        nn.BatchNorm1d(shared_dim),
        nn.ReLU(),
        nn.Linear(shared_dim, prime_dim),
        nn.BatchNorm1d(prime_dim),
        nn.ReLU(),
    )


# feature_dim = 64 (cross) + prime_dim + prime_dim  →  prime_dim = (feature_dim - 64) // 2
prime_dim = (CONFIG["feature_dim"] - 64) // 2  # = 16
patch_unimodal(model.detection_module.uni_repre, prime_dim)
# uni_se must stay at 64 to match SEAttentionModule(text_dim=64, image_dim=64, correlation_dim=64)
patch_unimodal(model.detection_module.uni_se, 64)

print(f"✅ Model patched for ResNet50/PhoBERT dimensions")
print(f"   uni_repre prime_dim={prime_dim}, uni_se prime_dim=64")
print(
    f"   feature_dim={CONFIG['feature_dim']} = 64 (cross) + {prime_dim} + {prime_dim}"
)
print(f"   AmbiguityLearning.encoding NOT patched (unused in forward)")

In [ ]:
# Patch FastCNN with increased dropout (0.3)
def patched_fastcnn_forward(self, x):
    """Forward without permute - data already [B, 768, 512]."""
    x_out = []
    for module in self.fast_cnn:
        x_out.append(module(x).squeeze(-1))
    return torch.cat(x_out, 1)


def patch_cnn(m, input_dim, dropout=0.3):
    """Recreate FastCNN with higher dropout."""
    channel = 32
    kernel_size = (1, 2, 4, 8)

    new_cnn = nn.ModuleList()
    for kernel in kernel_size:
        new_cnn.append(
            nn.Sequential(
                nn.Conv1d(input_dim, channel, kernel_size=kernel),
                nn.BatchNorm1d(channel),
                nn.ReLU(),
                nn.Dropout(dropout),  # Increased dropout
                nn.AdaptiveMaxPool1d(1),
            )
        )
    m.fast_cnn = new_cnn

    # Patch forward
    import types

    m.forward = types.MethodType(patched_fastcnn_forward, m)


# Apply to all FastCNN modules
patch_cnn(
    model.similarity_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"]
)
patch_cnn(
    model.detection_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"]
)
patch_cnn(
    model.detection_module.ambiguity_module.encoding.shared_text_encoding,
    TEXT_EMBED,
    CONFIG["dropout"],
)

# Move to device
model = model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model ready with {n_params:,} trainable parameters")
print(f"   Dropout: {CONFIG['dropout']}")

In [ ]:
def make_pair_sim(text, image, hard_negative_ratio=0.3):
    """
    Generate balanced real/fake pairs from a full batch of matched (real) data.

    Hard negatives: cosine-similarity mining — picks the most visually similar
    (but wrong) image for each text, making fakes genuinely confusing.
    Random negatives: cyclic shift with random offset (guaranteed no self-pairing).

    Returns:
        text_all:   [2B, 768, 512]
        image_all:  [2B, 2048]
        labels:     [2B]  — 1=real, 0=fake
        t_anchor:   [B, 768, 512]
        i_positive: [B, 2048]
        i_negative: [B, 2048]
    """
    B = text.size(0)
    device = text.device

    # Hard negatives: find most similar (confusing) wrong image via cosine sim
    with torch.no_grad():
        img_norm = F.normalize(image, dim=1)  # [B, 2048]
        sim = img_norm @ img_norm.T  # [B, B]
        sim.fill_diagonal_(-1e9)  # exclude self
        hard_idx = sim.argmax(dim=1)  # [B] — most similar != self

    # Random negatives: cyclic shift, guaranteed no self-pairing
    offset = random.randint(1, max(1, B - 1))
    rand_idx = (torch.arange(B, device=device) + offset) % B

    # Mix: first n_hard use similarity mining, rest use random shift
    n_hard = int(B * hard_negative_ratio)
    neg_idx = rand_idx.clone()
    neg_idx[:n_hard] = hard_idx[:n_hard]

    image_fake = image[neg_idx]  # [B, 2048] — new tensor via indexing

    # Detection: full 2B batch with balanced labels
    text_all = torch.cat([text, text], dim=0)
    image_all = torch.cat([image, image_fake], dim=0)
    labels = torch.cat(
        [
            torch.ones(B, dtype=torch.long, device=device),  # real = 1
            torch.zeros(B, dtype=torch.long, device=device),  # fake = 0
        ],
        dim=0,
    )

    # Similarity triplet
    t_anchor = text
    i_positive = image
    i_negative = image_fake

    return text_all, image_all, labels, t_anchor, i_positive, i_negative


print("✅ make_pair_sim defined (cosine hard-negative mining)")
print("   - Hard negatives: most similar wrong image (not random roll)")
print("   - Random negatives: cyclic shift, no self-pairing")
print("   - labels: 1=real, 0=fake  (balanced 50/50)")

In [ ]:
# Optimizers with weight decay
wd = CONFIG["weight_decay"]

optim_sim = torch.optim.Adam(
    model.similarity_module.parameters(), lr=CONFIG["lr"], weight_decay=wd
)

optim_clip = torch.optim.AdamW(
    model.clip_module.parameters(), lr=1e-3, weight_decay=5e-4
)

optim_det = torch.optim.Adam(
    model.detection_module.parameters(),
    lr=3e-4,  # reduced from 1e-3 — detection head needs slower, steadier updates
    weight_decay=wd,
)

print(f"✅ Optimizers created")
print(f"   optim_sim  lr=1e-3,  weight_decay={wd}")
print(f"   optim_clip lr=1e-3,  weight_decay=5e-4")
print(f"   optim_det  lr=3e-4,  weight_decay={wd}  ← reduced")

In [ ]:
# Cosine annealing scheduler with warmup
class WarmupCosineScheduler:
    """Warmup then cosine annealing."""

    def __init__(self, optimizer, warmup_steps, total_steps, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr
        self.current_step = 0
        self.base_lrs = [group["lr"] for group in optimizer.param_groups]

    def step(self):
        self.current_step += 1
        for i, base_lr in enumerate(self.base_lrs):
            if self.current_step < self.warmup_steps:
                lr = base_lr * self.current_step / self.warmup_steps
            else:
                progress = (self.current_step - self.warmup_steps) / (
                    self.total_steps - self.warmup_steps
                )
                lr = self.min_lr + (base_lr - self.min_lr) * 0.5 * (
                    1 + math.cos(math.pi * progress)
                )
            self.optimizer.param_groups[i]["lr"] = lr

    def get_lr(self):
        return [group["lr"] for group in self.optimizer.param_groups]


# Derived from CONFIG — stays in sync if max_epochs changes
steps_per_epoch = len(loaders["train"])
total_steps = steps_per_epoch * CONFIG["max_epochs"]
warmup_steps = int(0.01 * total_steps)  # 1% warmup

sch_sim = WarmupCosineScheduler(optim_sim, warmup_steps, total_steps)
sch_clip = WarmupCosineScheduler(optim_clip, warmup_steps, total_steps)
sch_det = WarmupCosineScheduler(optim_det, warmup_steps, total_steps)

print(f"✅ Schedulers created")
print(f"   max_epochs={CONFIG['max_epochs']}, total_steps={total_steps}")
print(f"   warmup_steps={warmup_steps} ({warmup_steps/steps_per_epoch:.1f} epochs)")
print(f"   LR decays to min_lr=1e-6 exactly at epoch {CONFIG['max_epochs']}")

In [ ]:
# Loss functions
loss_cos = nn.CosineEmbeddingLoss(margin=0.2)

# Label smoothing for detection loss
loss_ce = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])

loss_kl = nn.KLDivLoss(reduction="batchmean")


def soft_xe(logits, soft_target):
    return -(soft_target * F.log_softmax(logits, 1)).sum() / logits.size(0)


print(f"✅ Loss functions created")
print(f"   Label smoothing: {CONFIG['label_smoothing']}")
print(f"   Gradient clipping: {CONFIG['grad_clip']}")

In [ ]:
def run_epoch(loader, train=True, epoch_num=None):
    if train:
        model.train()
    else:
        model.eval()

    tot_loss, tot_ok, tot_n = 0.0, 0, 0
    all_y, all_p = [], []
    epoch_sim_loss, epoch_clip_loss, epoch_det_loss = 0.0, 0.0, 0.0
    num_batches = 0
    label_counts = {0: 0, 1: 0}

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for text, image in tqdm(loader, desc="Train" if train else "Eval"):
            text = text.to(DEVICE)
            image = image.to(DEVICE)
            bs = text.size(0)

            text_all, image_all, labels, t_anchor, i_positive, i_negative = (
                make_pair_sim(text, image)
            )

            if train:
                optim_sim.zero_grad()
                optim_clip.zero_grad()
                optim_det.zero_grad()

            # Task 1a: Similarity (cosine embedding loss)
            # Cache positive-pair features (reused for CLIP soft-targets and detection)
            ts_pos, is_pos, _ = model.similarity_module(t_anchor, i_positive)
            ts_neg, is_neg, _ = model.similarity_module(t_anchor, i_negative)
            t_cat = torch.cat([ts_pos, ts_neg])
            i_cat = torch.cat([is_pos, is_neg])
            y_cos = torch.cat(
                [
                    torch.ones(ts_pos.size(0), device=DEVICE),
                    -torch.ones(ts_neg.size(0), device=DEVICE),
                ]
            )
            ls = loss_cos(t_cat, i_cat, y_cos)
            epoch_sim_loss += ls.item()

            # Task 1b: CLIP (B real pairs)
            text_pooled = text.mean(dim=2)  # [B, 768]
            ie, te = model.clip_module(image, text_pooled)
            logits = ie @ te.T * math.exp(0.07)
            ids = torch.arange(bs, device=DEVICE)

            # Reuse cached ts_pos/is_pos for soft-targets (same as similarity_module(text, image))
            soft_m = is_pos @ ts_pos.T * math.exp(0.07)

            lc = (loss_ce(logits, ids) + loss_ce(logits.T, ids)) / 2
            ls2 = (
                soft_xe(logits, F.softmax(soft_m, 1))
                + soft_xe(logits.T, F.softmax(soft_m.T, 1))
            ) / 2
            l_clip = lc + 0.2 * ls2
            epoch_clip_loss += l_clip.item()

            # Task 2: Detection — build ts_all/is_all from cached pos + neg features
            # instead of re-encoding the full 2B batch through similarity_module
            ts_all = torch.cat([ts_pos, ts_neg], dim=0)
            is_all = torch.cat([is_pos, is_neg], dim=0)
            det, attn, skl = model.detection_module(text_all, image_all, ts_all, is_all)
            ld = loss_ce(det, labels) + 0.1 * loss_kl(
                F.log_softmax(attn, 1), F.softmax(skl, 1)
            )
            epoch_det_loss += ld.item()

            if train:
                (ls + l_clip + ld).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
                optim_sim.step()
                optim_clip.step()
                optim_det.step()
                sch_sim.step()
                sch_clip.step()
                sch_det.step()

            # Metrics on 2B detection batch
            pred = det.argmax(1)
            tot_ok += pred.eq(labels).sum().item()
            tot_n += labels.size(0)
            tot_loss += ld.item() * labels.size(0)
            all_y.extend(labels.cpu().numpy())
            all_p.extend(pred.cpu().numpy())
            num_batches += 1

            if epoch_num == 1:
                label_counts[0] += (labels == 0).sum().item()
                label_counts[1] += (labels == 1).sum().item()

    metrics = {
        "loss": tot_loss / tot_n,
        "acc": tot_ok / tot_n,
        "sim_loss": epoch_sim_loss / num_batches,
        "clip_loss": epoch_clip_loss / num_batches,
        "det_loss": epoch_det_loss / num_batches,
    }

    if epoch_num == 1 and train:
        total_labels = label_counts[0] + label_counts[1]
        print(
            f"  📊 Label dist: real={label_counts[1]} ({label_counts[1]/total_labels:.0%}), "
            f"fake={label_counts[0]} ({label_counts[0]/total_labels:.0%})"
        )

    return metrics, all_y, all_p


print("✅ run_epoch defined")
print("   Optimizations:")
print("   - Cached similarity_module(text, image) → reused for CLIP soft-targets")
print(
    "   - Built ts_all/is_all from cached pos+neg → eliminated redundant 2B forward pass"
)
print("   - 2 similarity_module calls per batch (was 4)")

In [ ]:
# MLflow setup
EXPERIMENT_NAME = "coolant-vifactcheck-anti-overfitting"
mlflow.set_experiment(EXPERIMENT_NAME)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"coolant-vifactcheck-{timestamp}"
mlflow.start_run(run_name=run_name)

mlflow.log_params(CONFIG)
print(f"✅ MLflow run started: {run_name}")

In [ ]:
# Training loop with early stopping
best_val_loss = float("inf")
best_val_acc = 0.0
patience_counter = 0
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

print(f"🚀 Starting training for max {CONFIG['max_epochs']} epochs...")
print(f"   Early stopping patience: {CONFIG['patience']}")
print(f"   Best model based on: val_loss\n")

for epoch in range(1, CONFIG["max_epochs"] + 1):
    # Train
    train_metrics, _, _ = run_epoch(loaders["train"], train=True, epoch_num=epoch)

    # Validate
    val_metrics, _, _ = run_epoch(loaders["dev"], train=False, epoch_num=epoch)

    # Store history
    history["train_loss"].append(train_metrics["loss"])
    history["train_acc"].append(train_metrics["acc"])
    history["val_loss"].append(val_metrics["loss"])
    history["val_acc"].append(val_metrics["acc"])

    # Log to MLflow
    mlflow.log_metrics(
        {
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["acc"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
            "lr_sim": optim_sim.param_groups[0]["lr"],
        },
        step=epoch,
    )

    # Print progress
    print(f"Epoch [{epoch:02d}/{CONFIG['max_epochs']}] ")
    print(f"  Train: loss={train_metrics['loss']:.4f} acc={train_metrics['acc']:.4f}")
    print(f"  Val:   loss={val_metrics['loss']:.4f} acc={val_metrics['acc']:.4f}")

    # Checkpoint based on val_loss
    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        best_val_acc = val_metrics["acc"]
        patience_counter = 0

        # Save best model
        checkpoint_path = SAVE_DIR / "best_model.pth"
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optim_sim_state_dict": optim_sim.state_dict(),
                "optim_clip_state_dict": optim_clip.state_dict(),
                "optim_det_state_dict": optim_det.state_dict(),
                "val_loss": best_val_loss,
                "val_acc": best_val_acc,
                "config": CONFIG,
            },
            checkpoint_path,
        )
        mlflow.log_artifact(str(checkpoint_path), artifact_path="checkpoints")
        print(f"  ✓ New best val_loss={best_val_loss:.4f}, saved checkpoint")
    else:
        patience_counter += 1
        print(f"  → No improvement ({patience_counter}/{CONFIG['patience']})")

    # Early stopping
    if patience_counter >= CONFIG["patience"]:
        print(f"\n🛑 Early stopping triggered at epoch {epoch}")
        break

    print()

print(f"\n✅ Training complete!")
print(f"   Best val_loss: {best_val_loss:.4f}")
print(f"   Best val_acc: {best_val_acc:.4f}")

In [ ]:
# Load best model
checkpoint = torch.load(SAVE_DIR / "best_model.pth", map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"✅ Loaded best model from epoch {checkpoint['epoch']}")
print(f"   Val loss: {checkpoint['val_loss']:.4f}")
print(f"   Val acc: {checkpoint['val_acc']:.4f}")

In [ ]:
# Test evaluation
print("\n🔍 Evaluating on test set...")
test_metrics, test_y, test_p = run_epoch(loaders["test"], train=False)

print(f"\n✅ Test Results:")
print(f"   Loss: {test_metrics['loss']:.4f}")
print(f"   Accuracy: {test_metrics['acc']:.4f}")

# Classification report
report = classification_report(test_y, test_p, target_names=["Real", "Fake"], digits=4)
print("\nClassification Report:")
print(report)

# Confusion matrix
cm = confusion_matrix(test_y, test_p)
print("\nConfusion Matrix:")
print(cm)

# Log to MLflow
mlflow.log_metrics(
    {
        "test_loss": test_metrics["loss"],
        "test_acc": test_metrics["acc"],
    }
)

# Save report
report_path = SAVE_DIR / "test_report.txt"
with open(report_path, "w") as f:
    f.write("Test Classification Report\n")
    f.write("=" * 50 + "\n\n")
    f.write(report)
    f.write("\n\nConfusion Matrix:\n")
    f.write(str(cm))

mlflow.log_artifact(str(report_path), artifact_path="results")
print("\n✅ Test report saved")

In [ ]:
# End MLflow run
mlflow.end_run()
print("✅ MLflow run ended")
print(f"\nExperiment: {EXPERIMENT_NAME}")
print(f"Run: {run_name}")

In [ ]:
# Summary
print(f"{'='*60}")
print("📊 TRAINING SUMMARY")
print(f"{'='*60}\n")

print("Anti-overfitting techniques applied:")
print(f"  • Dropout: {CONFIG['dropout']}")
print(f"  • Weight decay: {CONFIG['weight_decay']}")
print(f"  • Label smoothing: {CONFIG['label_smoothing']}")
print(f"  • Gradient clipping: {CONFIG['grad_clip']}")
print(f"  • Early stopping patience: {CONFIG['patience']}")
print(f"  • Cosine annealing + warmup")
print()
print("Final Results:")
print(f"  Best val_loss: {best_val_loss:.4f}")
print(f"  Best val_acc: {best_val_acc:.4f}")
print(f"  Test acc: {test_metrics['acc']:.4f}")
print()
print(f"{'='*60}")
print("✅ Training complete!")
print(f"{'='*60}")